In [3]:
import pandas as pd
import numpy as np

# ML
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix


In [5]:
# -----------------------------
# 1. LOAD DATA
# -----------------------------
file_path = r"C:\Users\alina\Desktop\Portfolio\LLyods-Banking-Group-Risk-Customer-Crunch-Prediction\data\Customer_Churn_Data_Large.xlsx"

demographics = pd.read_excel(file_path, sheet_name="Customer_Demographics")
transactions = pd.read_excel(file_path, sheet_name="Transaction_History")
service = pd.read_excel(file_path, sheet_name="Customer_Service")
online = pd.read_excel(file_path, sheet_name="Online_Activity")
churn = pd.read_excel(file_path, sheet_name="Churn_Status")
# -----------------------------

In [8]:
# -----------------------------
# 2. MERGE DATA
# -----------------------------
df = demographics.merge(churn, on="CustomerID")
df = df.merge(online, on="CustomerID", how="left")
df.head()

,CustomerID,Age,Gender,MaritalStatus,IncomeLevel,ChurnStatus,LastLoginDate,LoginFrequency,ServiceUsage
0,1,62,M,Single,Low,0,2023-10-21,34,Mobile App
1,2,65,M,Married,Low,1,2023-12-05,5,Website
2,3,18,M,Single,Low,0,2023-11-15,3,Website
3,4,21,M,Widowed,Low,0,2023-08-25,2,Website
4,5,21,M,Divorced,Medium,0,2023-10-27,41,Website


In [ ]:
# -----------------------------
# 3. FEATURE ENGINEERING
# -----------------------------

# Transaction features
txn_features = transactions.groupby("CustomerID").agg({
    "AmountSpent": ["sum", "mean", "count"]
}).reset_index()

txn_features.columns = ["CustomerID", "TotalSpend", "AvgSpend", "TxnCount"]

# Service features
service_features = service.groupby("CustomerID").agg({
    "InteractionType": "count"
}).reset_index()

service_features.columns = ["CustomerID", "ServiceInteractions"]

# Merge features
df = df.merge(txn_features, on="CustomerID", how="left")
df = df.merge(service_features, on="CustomerID", how="left")

# Fill missing values
df.fillna(0, inplace=True)

# -----------------------------
# 4. PREPROCESSING
# -----------------------------

# Drop ID
df = df.drop("CustomerID", axis=1)

# Separate features and target
X = df.drop("ChurnStatus", axis=1)
y = df["ChurnStatus"]

# Categorical & numerical columns
cat_cols = ["Gender", "MaritalStatus", "IncomeLevel", "ServiceUsage"]
num_cols = ["Age", "LoginFrequency", "TotalSpend", "AvgSpend", "TxnCount", "ServiceInteractions"]

# Encoding
X = pd.get_dummies(X, columns=cat_cols)

# Scaling
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# -----------------------------
# 5. TRAIN TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# -----------------------------
# 6. MODEL TRAINING
# -----------------------------
rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(rf, param_grid, cv=5, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)

model = grid.best_estimator_

# -----------------------------
# 7. EVALUATION
# -----------------------------
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))